# 34. Neural Networks: Convolutional Neural Networks (CNNs)

## Algorithm Category
**Type**: Neural Networks - Deep Learning  
**Complexity**: High  
**Use Case**: Image classification, object detection, computer vision tasks

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand convolutional layers and their operations
- Implement CNNs using PyTorch
- Understand pooling, padding, and stride
- Visualize learned filters and feature maps
- Apply CNNs to image classification
- Understand transfer learning with pre-trained models

## Historical Context

CNNs were inspired by biological vision:
- LeCun, Y., et al. (1998): "Gradient-based learning applied to document recognition"
- LeNet-5 (1998): First successful CNN for digit recognition
- AlexNet (2012): Breakthrough in ImageNet competition
- Foundation for modern computer vision

**Key Papers/References:**
- LeCun, Y., et al. (1998). "Gradient-based learning applied to document recognition"
- Krizhevsky, A., et al. (2012). "ImageNet classification with deep convolutional neural networks"

## When to Use CNNs

CNNs are appropriate when:
- Working with image data
- Need spatial feature extraction
- Object detection and recognition
- Computer vision tasks
- When data has grid-like structure (images, time series)
- Transfer learning from pre-trained models

## Theory & Mechanics

### Mathematical Foundation

**Convolution Operation:**
$$(f * g)(x, y) = \sum_{i} \sum_{j} f(i, j) \cdot g(x-i, y-j)$$

**Convolutional Layer:**
$$h_{i,j} = \sum_{m} \sum_{n} x_{i+m, j+n} \cdot w_{m,n} + b$$

**Max Pooling:**
$$h_{i,j} = \max_{m,n \in \text{pool}} x_{i+m, j+n}$$

### Key Components

1. **Convolutional Layers**
   - Apply filters (kernels) to input
   - Detect local patterns (edges, textures)
   - Share weights across spatial locations
   - Reduce parameters compared to fully connected

2. **Pooling Layers**
   - Downsample feature maps
   - Reduce spatial dimensions
   - Max pooling: take maximum value
   - Average pooling: take average value

3. **Activation Functions**
   - ReLU: Introduces non-linearity
   - Applied after convolution

4. **Fully Connected Layers**
   - Final classification layers
   - Flatten feature maps

### How It Works

1. **Convolution**: Apply filters to detect features
2. **Activation**: Apply ReLU for non-linearity
3. **Pooling**: Downsample to reduce size
4. **Repeat**: Stack multiple conv layers
5. **Flatten**: Convert to 1D for classification
6. **Dense**: Final classification layers

### Key Hyperparameters

- **filters**: Number of convolutional filters
- **kernel_size**: Size of convolution kernel
- **stride**: Step size for convolution
- **padding**: Border handling ('same', 'valid')
- **pool_size**: Size of pooling window
- **dropout**: Regularization rate

### Advantages

- Translation invariant
- Parameter sharing (efficient)
- Hierarchical feature learning
- Excellent for images
- Can use transfer learning

### Limitations

- Requires large datasets
- Computationally expensive
- Black box (hard to interpret)
- Sensitive to input size
- Requires GPU for training


## Implementation

Let's implement CNNs using PyTorch for image classification.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Libraries imported successfully!")


In [ ]:
# Simple CNN Architecture
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # Assuming 28x28 input -> 7x7 after pooling
        self.fc2 = nn.Linear(128, num_classes)
        # Activation
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        
        # Conv block 2
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        
        # Flatten
        x = x.view(-1, 64 * 7 * 7)
        
        # Fully connected
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

print("CNN model defined!")


In [ ]:
# Load MNIST dataset (using a subset for faster training)
try:
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X, y = mnist.data, mnist.target.astype(int)
    print(f"MNIST dataset loaded: {X.shape}")
except Exception as e:
    print(f"Error loading MNIST: {e}")
    print("Creating synthetic image data for demonstration...")
    # Create synthetic 28x28 images
    X = np.random.rand(1000, 784).astype(np.float32)
    y = np.random.randint(0, 10, 1000)

# Use subset for faster training
n_samples = min(5000, len(X))
X = X[:n_samples]
y = y[:n_samples]

# Reshape to images (28x28)
X_images = X.reshape(-1, 28, 28)

# Normalize
X_images = X_images / 255.0

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_images, y, test_size=0.2, random_state=42
)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train).unsqueeze(1)  # Add channel dimension
X_test_tensor = torch.FloatTensor(X_test).unsqueeze(1)
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)

print(f"Training set: {X_train_tensor.shape}")
print(f"Test set: {X_test_tensor.shape}")
print(f"Number of classes: {len(np.unique(y))}")


In [ ]:
# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Initialize model
model = SimpleCNN(num_classes=len(np.unique(y))).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


## Training

Let's train the CNN model.


In [ ]:
# Training loop
num_epochs = 5
train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%")

print("\nTraining complete!")


## Evaluation

Let's evaluate the model on test data.


In [ ]:
# Evaluate on test set
model.eval()
test_correct = 0
test_total = 0
all_predictions = []
all_targets = []

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = torch.max(output.data, 1)
        test_total += target.size(0)
        test_correct += (predicted == target).sum().item()
        all_predictions.extend(predicted.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

test_accuracy = 100 * test_correct / test_total
print(f"Test Accuracy: {test_accuracy:.2f}%")

# Classification report
print("\nClassification Report:")
print(classification_report(all_targets, all_predictions))


## Visualization

Let's visualize training progress and sample predictions.


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'o-')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_accuracies, 's-', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualize sample predictions
model.eval()
with torch.no_grad():
    sample_data = X_test_tensor[:8].to(device)
    sample_targets = y_test_tensor[:8]
    sample_output = model(sample_data)
    _, sample_predicted = torch.max(sample_output, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i in range(8):
    img = sample_data[i].cpu().squeeze().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'True: {sample_targets[i]}, Pred: {sample_predicted[i].item()}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()


## Feature Maps Visualization

Let's visualize what the CNN learns by examining feature maps.


In [ ]:
# Visualize learned filters (first convolutional layer)
with torch.no_grad():
    filters = model.conv1.weight.data.cpu().numpy()
    
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i in range(min(32, filters.shape[0])):
    row = i // 8
    col = i % 8
    filter_img = filters[i, 0]  # First channel
    axes[row, col].imshow(filter_img, cmap='gray')
    axes[row, col].axis('off')
    axes[row, col].set_title(f'Filter {i}', fontsize=8)

plt.suptitle('Learned Filters (First Convolutional Layer)', fontsize=14)
plt.tight_layout()
plt.show()

# Visualize feature maps for a sample image
model.eval()
sample_img = X_test_tensor[0:1].to(device)

# Hook to capture feature maps
feature_maps = []
def hook_fn(module, input, output):
    feature_maps.append(output.detach().cpu().numpy())

hook = model.conv1.register_forward_hook(hook_fn)
with torch.no_grad():
    _ = model(sample_img)
hook.remove()

if feature_maps:
    fm = feature_maps[0][0]  # First batch, all channels
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    for i in range(min(32, fm.shape[0])):
        row = i // 8
        col = i % 8
        axes[row, col].imshow(fm[i], cmap='viridis')
        axes[row, col].axis('off')
    
    plt.suptitle('Feature Maps (First Convolutional Layer)', fontsize=14)
    plt.tight_layout()
    plt.show()


## Validation & Testing

Let's validate the model performance.


In [ ]:
# Assertions
assert test_accuracy > 50, "CNN should perform better than random"
assert len(train_losses) == num_epochs, "Should have trained for all epochs"
print("\n✓ Validation checks passed")

# Compare with simple MLP (for reference)
print("\nNote: CNNs are specifically designed for image data.")
print("They use parameter sharing and local connectivity,")
print("making them more efficient than fully connected networks for images.")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Convolutional Layers**
   - Apply filters to detect local patterns
   - Share weights across spatial locations
   - Translation invariant
   - Efficient parameter usage

2. **Pooling Layers**
   - Downsample feature maps
   - Reduce spatial dimensions
   - Max pooling: preserves strongest features
   - Helps with overfitting

3. **CNN Architecture**
   - Stack of conv + activation + pooling
   - Feature extraction layers
   - Classification layers (fully connected)
   - Hierarchical feature learning

4. **Key Components**
   - **Filters/Kernels**: Detect features
   - **Stride**: Step size
   - **Padding**: Border handling
   - **Feature maps**: Output of convolutions

### When to Use CNNs

✅ **Good for:**
- Image classification
- Object detection
- Computer vision tasks
- Data with spatial structure
- Transfer learning
- When you need translation invariance

❌ **Not ideal for:**
- Tabular data (use MLPs)
- Sequential data (use RNNs)
- Very small datasets
- When interpretability is critical
- Real-time applications (can be slow)

### Next Steps

- Explore **Transfer Learning** with pre-trained models
- Try **Data Augmentation** to improve performance
- Experiment with **different architectures** (ResNet, VGG, etc.)
- Apply to **object detection** tasks
- Use **Batch Normalization** for better training
